# 06 — TTA (Test Time Augmentation) 평가

**목적**: 이미 학습된 Baseline 모델에 **추론 시 TTA 적용** → mIoU 향상 측정.

**학습 X**. ckpt 그대로 활용. **~30분 소요**.

**TTA 전략**: Original + Hflip + Rot±10° = 4개 logits 평균.

**⚠️ mediapipe는 이번 노트북에서 설치 안 함** (Colab의 numpy/pandas 와 충돌 회피).
LM-guided 결과는 Phase 3 기록값 (0.6831) 재사용.

**Week 3 DoD**:
- Baseline no TTA vs Baseline + TTA mIoU 비교
- 발표 슬라이드 8번 데이터 확보

## 1. 환경 셋업 (mediapipe 제외)

In [ ]:
!nvidia-smi

In [ ]:
# mediapipe 제외 — numpy 다운그레이드 방지
!pip install --quiet segmentation-models-pytorch albumentations datasets pyyaml tqdm

In [ ]:
import torch, segmentation_models_pytorch as smp
print('torch:', torch.__version__, '  cuda:', torch.cuda.is_available())
print('smp:', smp.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/cv-final'
import sys, os
sys.path.insert(0, PROJECT_ROOT)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. 체크포인트 경로 자동 탐색

In [ ]:
CKPT_DIR = f'{PROJECT_ROOT}/project/segmentation/checkpoints'

import glob
print('=== Drive에 있는 체크포인트 ===')
ckpts = sorted(glob.glob(f'{CKPT_DIR}/*.pt'))
for p in ckpts:
    size_mb = os.path.getsize(p) / 1024 / 1024
    print(f'  {os.path.basename(p)}  ({size_mb:.1f} MB)')

# Baseline ckpt 자동 선택 (우선순위: baseline_best > v1)
candidates = [
    f'{CKPT_DIR}/unet_baseline_best.pt',
    f'{CKPT_DIR}/unet_v1.pt',
]
BASELINE_CKPT = None
for c in candidates:
    if os.path.exists(c):
        BASELINE_CKPT = c
        break

if BASELINE_CKPT is None:
    raise FileNotFoundError(
        f'Baseline ckpt 없음. {CKPT_DIR}/ 에 unet_baseline_best.pt 또는 unet_v1.pt 필요.'
    )

print(f'\n✅ 사용할 baseline ckpt: {BASELINE_CKPT}')

## 3. Validation 데이터셋 (3채널 baseline)

In [ ]:
from project.segmentation import CelebAMaskHQDataset, get_val_transform
from torch.utils.data import DataLoader

BATCH = 16
CACHE = '/content/cv-final-cache'

val_ds = CelebAMaskHQDataset(
    split='val',
    transform=get_val_transform(size=256, with_heatmap=False),
    cache_dir=CACHE,
    use_landmark=False,
    verbose=True,
)
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)
print(f'val: {len(val_ds)} samples')

## 4. Baseline 모델 로드

In [ ]:
from project.segmentation.unet import build_unet

model_baseline = build_unet(num_classes=6, in_channels=3, encoder_weights=None)
model_baseline.load_state_dict(torch.load(BASELINE_CKPT, map_location=device))
model_baseline.to(device).eval()
print('✅ Baseline U-Net 로드 완료')

## 5. 평가 — Baseline no TTA (기준선)

In [ ]:
from project.segmentation.train import evaluate

print('평가 시작 (no TTA)...')
result_no_tta = evaluate(model_baseline, val_dl, device, num_classes=6)
print()
print(f"  Baseline no TTA → mIoU: {result_no_tta['miou']:.4f}, Dice: {result_no_tta['dice']:.4f}")

## 6. 평가 — Baseline + TTA (★ 핵심)

약 25분 소요 (4배 느린 추론).

In [ ]:
from project.segmentation.tta import evaluate_with_tta

TTA_AUGS = ['original', 'hflip', 'rot+', 'rot-']
print(f'TTA augmentations: {TTA_AUGS}')
print('평가 시작 (TTA)... ~25분')
result_tta = evaluate_with_tta(
    model_baseline, val_dl, device, num_classes=6, augmentations=TTA_AUGS
)
print()
print(f"  Baseline + TTA → mIoU: {result_tta['miou']:.4f}, Dice: {result_tta['dice']:.4f}")

## 7. 최종 비교표 (Phase 3 LM 결과 포함)

In [ ]:
# Phase 3 기록값 (재학습 안 함)
LM_NO_TTA_KNOWN = 0.6831

base_no = result_no_tta['miou']
base_tta = result_tta['miou']

print('═' * 72)
print('   강화 단계별 효과 (Val mIoU)')
print('═' * 72)
print(f'  Baseline U-Net           :  {base_no:.4f}                  ← 기준선')
print(f'  + LM Heatmap             :  {LM_NO_TTA_KNOWN:.4f}   (Δ {(LM_NO_TTA_KNOWN-base_no)*100:+.2f}%p)  [Phase 3]')
print(f'  + TTA (★)                :  {base_tta:.4f}   (Δ {(base_tta-base_no)*100:+.2f}%p)  [이번 측정]')
print('═' * 72)

tta_gain = (base_tta - base_no) * 100
if tta_gain >= 2.0:
    print(f'  ✅ TTA 효과 큼 (+{tta_gain:.2f}%p) — 발표 자료에 단계별 표 포함 가능')
elif tta_gain >= 0.5:
    print(f'  ⚠️  TTA 효과 보통 (+{tta_gain:.2f}%p) — 추가 강화 검토 (Attention/Boundary)')
else:
    print(f'  🟡 TTA 효과 미미 (+{tta_gain:.2f}%p) — Architecture 비교 또는 다른 방향')

print()
print('Note: LM + TTA 조합은 mediapipe 환경 충돌로 이번 세션에서는 측정 안 함.')
print('     필요 시 다음 세션에서 mediapipe 설치 후 별도 평가.')

## 8. 시각화 — no TTA vs TTA 예측 비교 (5장)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from project.segmentation.tta import predict_with_tta

ds_viz = CelebAMaskHQDataset(
    split='val',
    transform=get_val_transform(size=256, with_heatmap=False),
    subset_size=5, cache_dir=CACHE, use_landmark=False,
)

fig, axes = plt.subplots(5, 4, figsize=(16, 18))
for i in range(5):
    img, gt_mask = ds_viz[i]
    rgb = img.permute(1, 2, 0).numpy()
    rgb = rgb * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    rgb = rgb.clip(0, 1)

    # Baseline no TTA
    with torch.no_grad():
        pred_no = model_baseline(img.unsqueeze(0).to(device)).argmax(dim=1).squeeze(0).cpu().numpy()

    # Baseline + TTA
    avg_logits = predict_with_tta(model_baseline, img, device=device,
                                   augmentations=TTA_AUGS)
    pred_tta = avg_logits.argmax(dim=0).cpu().numpy()

    axes[i, 0].imshow(rgb); axes[i, 0].set_title('Input'); axes[i, 0].axis('off')
    axes[i, 1].imshow(gt_mask.numpy(), cmap='tab10', vmin=0, vmax=5)
    axes[i, 1].set_title('Ground Truth'); axes[i, 1].axis('off')
    axes[i, 2].imshow(pred_no, cmap='tab10', vmin=0, vmax=5)
    axes[i, 2].set_title('Baseline (no TTA)'); axes[i, 2].axis('off')
    axes[i, 3].imshow(pred_tta, cmap='tab10', vmin=0, vmax=5)
    axes[i, 3].set_title('Baseline + TTA ★'); axes[i, 3].axis('off')

plt.tight_layout()
os.makedirs(f'{PROJECT_ROOT}/data/outputs', exist_ok=True)
plt.savefig(f'{PROJECT_ROOT}/data/outputs/tta_comparison.png', dpi=150)
plt.show()

## 9. 완료 체크리스트

- [ ] 체크포인트 자동 발견 (`unet_baseline_best.pt` 또는 `unet_v1.pt`)
- [ ] Baseline no TTA mIoU 측정 (~0.68 예상)
- [ ] Baseline + TTA mIoU 측정 (+1~2%p 향상 예상)
- [ ] 비교표 출력 (Baseline / +LM / +TTA)
- [ ] 시각화 5장 저장 (`data/outputs/tta_comparison.png`)

**다음 단계**:
- TTA 효과 +2%p 이상 → 발표 자료 확정, Streamlit 진입
- 미달 → Attention Gate 추가 강화
- LM + TTA 조합은 별도 세션에서 (mediapipe 환경 분리)